# EasyImaging — Measurement viewer

Pick a **SciTiff** file in the file explorer on the left.
The file is loaded into a `Measurement`, and its image is plotted on the right.

In [ ]:
%matplotlib widget

import html
from pathlib import Path

import ipywidgets as widgets
import pooch

from easyimaging import Measurement

In [ ]:
TIFF_SUFFIXES = ('.tif', '.tiff')
PARENT_LABEL = '📁 ..'


def default_directory() -> Path:
    """Start the explorer in the cache folder of the bundled example data, if it has been downloaded."""
    example_data = Path(pooch.os_cache('easyimaging'))
    return example_data if example_data.is_dir() else Path.cwd()


def directory_options(directory: Path) -> list[tuple[str, Path]]:
    """List the sub-folders and TIFF files of a folder as (label, path) pairs for a Select widget."""
    directories = []
    files = []
    for entry in sorted(directory.iterdir(), key=lambda path: path.name.lower()):
        try:
            if entry.name.startswith('.'):
                continue
            if entry.is_dir():
                directories.append((f'📁 {entry.name}', entry))
            elif entry.suffix.lower() in TIFF_SUFFIXES:
                files.append((f'🖼 {entry.name}', entry))
        except OSError:  # Unreadable entries are simply not offered
            continue
    return [(PARENT_LABEL, directory.parent)] + directories + files

In [ ]:
class MeasurementViewer:
    """A file explorer that loads the picked SciTiff file into a `Measurement` and plots it."""

    def __init__(self, directory: str | Path | None = None):
        self.directory = Path(directory) if directory is not None else default_directory()
        self.measurement = None
        self._muted = False  # Set while repopulating the explorer, so that it does not count as a pick

        self.location = widgets.HTML()
        self.explorer = widgets.Select(rows=18, layout=widgets.Layout(width='100%'))
        self.status = widgets.HTML()
        self.plot_area = widgets.VBox()
        self.explorer.observe(self._on_pick, names='value')

        self.widget = widgets.HBox([
            widgets.VBox(
                [self.location, self.explorer, self.status],
                layout=widgets.Layout(width='360px', flex='0 0 auto'),
            ),
            self.plot_area,
        ])

        self._show_directory(self.directory)
        self._set_status('Pick a SciTiff file to plot it.', 'grey')

    def _show_directory(self, directory: Path) -> None:
        """Fill the explorer with the content of a folder."""
        try:
            options = directory_options(directory)
        except OSError as error:
            self._set_status(f'Cannot open {html.escape(str(directory))}: {html.escape(str(error))}', 'crimson')
            return
        self.directory = directory
        self.location.value = f'<b>Folder</b><br><code>{html.escape(str(directory))}</code>'
        self._muted = True
        self.explorer.options = options
        self.explorer.index = None
        self._muted = False

    def _on_pick(self, change: dict) -> None:
        """Navigate into the picked folder, or load the picked file."""
        picked = change['new']
        if self._muted or picked is None:
            return
        if picked.is_dir():
            self._show_directory(picked)
        else:
            self._load(picked)

    def _load(self, filename: Path) -> None:
        """Load a SciTiff file into a `Measurement` and plot it."""
        self._set_status(f'Loading {html.escape(filename.name)} ...', 'grey')
        self.plot_area.children = ()
        try:
            self.measurement = Measurement.from_scitiff(filename=filename, display_name=filename.name)
        except Exception as error:
            self.measurement = None
            self._set_status(f'Could not load {html.escape(filename.name)}:<br>{html.escape(str(error))}', 'crimson')
            return
        # With the widget backend `plot` returns an interactive figure, which is itself an ipywidget.
        self.plot_area.children = (self.measurement.plot(),)
        self._set_status(f'Loaded {html.escape(filename.name)}', 'green')

    def _set_status(self, message: str, color: str) -> None:
        """Show a message underneath the file explorer."""
        self.status.value = f'<span style="color: {color}">{message}</span>'

In [ ]:
viewer = MeasurementViewer()
viewer.widget